### Plan
Baseline models:
* Pick on train step size and max iter for base models; 
    * I will look manually for such a step size, that flatlines as fast as possible, but still gives a smooth curve.
    * for the unregularized models. Regularized versions will use the same. 
    * Then I will cut the amount of iterations
* Train unregularized models.
    * Probabilistic models have threshold of 0.5
* then pick lambda for two regularized model by search on validation set. Pick pick the best model
* Then, obtain test scores - accuracy, recall, precision and most importantly, F1
* Obtain plots 
    * Barplot of Test F1 of the baseline models

extra contribution: 
* evaluate the impact of early stopping and threshold selection:
    * Keep the previous baseline.
    * 1. threshold selection intervention: tune the threshold, given lambda has been selected.
    * 2. early stopping prevention: do early stopping given 40 steps, when training and picking the best lambda. (40 is arbitraty, but OK)
    * then combined 1. and 2. : do early stopping (which affects the best lambda), and then pick the best threshold.
* plots:
    * barplot of test f1 of 4 options (as described above)

This notebook loads the final processed matrices cached by `data-cleaning.ipynb`.


## Load cached processed data

`fit.pkl`, `tune.pkl`, and `valid.pkl` contain processed feature matrices. No cleaning or transformation is rerun here.

In [4]:
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np

from p1.cache import cached_raw
from p1.config import INTERIM_DATA_DIR, RAW_DATA_DIR
from p1.modeling.models import LinearRegressGD, LinearRegressSGD, LogisticRegress
from p1.training import split_rows

CACHE = INTERIM_DATA_DIR / "fd_cleaning"
_, _, y_raw, _, _, _ = cached_raw(RAW_DATA_DIR, CACHE)

with (CACHE / "fit.pkl").open("rb") as handle:
    X_fit = pickle.load(handle)
with (CACHE / "tune.pkl").open("rb") as handle:
    X_tune = pickle.load(handle)
with (CACHE / "valid.pkl").open("rb") as handle:
    X_valid = pickle.load(handle)

y = (y_raw == 1).astype(float)
fit_idx, tune_idx, valid_idx = split_rows(y, seed=2026)
y_fit, y_tune, y_valid = y[fit_idx], y[tune_idx], y[valid_idx]
random.seed(2026)

assert X_fit.shape[0] == len(y_fit)
assert X_tune.shape[0] == len(y_tune)
assert X_valid.shape[0] == len(y_valid)
assert X_fit.shape[1] == X_tune.shape[1] == X_valid.shape[1]
print(f"Cached matrices: fit={X_fit.shape}, tune={X_tune.shape}, valid={X_valid.shape}")
print(f"Positive labels: {y.mean():.1%}")

Raw data: cache hit
Cached matrices: fit=(262508, 3809), tune=(32813, 3809), valid=(32814, 3809)
Positive labels: 8.8%


## Training curves

The following cells use only the processed fitting split. Adjust the step size and maximum iterations manually, then rerun the relevant cell.

In [5]:
# Materialize the cached sparse matrices for the dense model implementations.
X_fit = X_fit[:, :]
X_tune = X_tune[:, :]
X_valid = X_valid[:, :]
print(f"Dense matrices: fit={X_fit.shape}, tune={X_tune.shape}, valid={X_valid.shape}")

Dense matrices: fit=(262508, 3809), tune=(32813, 3809), valid=(32814, 3809)


In [ ]:
# Full-batch linear regression with gradient descent.
step_size_gd = 0.1
max_iters_gd = 100
linear_gd = LinearRegressGD(max_iters=max_iters_gd, gamma=step_size_gd, track_history=True)
linear_gd.fit(y_fit, X_fit)

plt.figure(figsize=(7, 4))
plt.plot(linear_gd.train_loss)
plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title(f"LinearRegressGD: gamma={step_size_gd}, max_iters={max_iters_gd}")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Stochastic linear regression with gradient descent.
step_size_sgd = 0.0001
max_iters_sgd = 25 * 1000
linear_sgd = LinearRegressSGD(max_iters=max_iters_sgd, gamma=step_size_sgd, track_history=True)
linear_sgd.fit(y_fit, X_fit)

plt.figure(figsize=(7, 4))
plt.plot(linear_sgd.train_loss)
plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title(f"LinearRegressSGD: gamma={step_size_sgd}, max_iters={max_iters_sgd}")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Unregularized logistic regression.
step_size_logistic = 0.1
max_iters_logistic = 100
logistic = LogisticRegress(max_iters=max_iters_logistic, gamma=step_size_logistic, track_history=True)
logistic.fit(y_fit, X_fit)

plt.figure(figsize=(7, 4))
plt.plot(logistic.train_loss)
plt.xlabel("Iteration")
plt.ylabel("Training negative log-likelihood")
plt.title(f"LogisticRegress: gamma={step_size_logistic}, max_iters={max_iters_logistic}")
plt.grid(alpha=0.3)
plt.show()